# 08 — Adapted Liu-ALNS (2024) External Comparator

Source: Liu, Sun, Duan, Liu (2024), *Scientific Reports* 14, 23809
(open-access). Only the serial destroy/repair/acceptance/cooling
algorithm is adapted -- NOT the paper's distributed Spark architecture.

**Approved adaptation** (full detail in `supplementary/LIU_ADAPTATION_MATRIX.md`):

| Component | Status |
|---|---|
| Random Removal | retained (same-vehicle suffix only) |
| Related Removal | retained (effectively distance-related in this setting -- audited) |
| Route Removal | **excluded** as non-transferable (locked vehicle assignment) |
| Random Repair | retained |
| Greedy Insertion | retained |
| Random-Criticality Repair | retained |
| Adaptive weighting | Ropke-Pisinger (2006) scheme, as cited by Liu et al. |
| Cooling | Liu et al.'s exact cosine-envelope formula |
| Locked prefix / fixed vehicle assignment | enforced structurally |
| Quantile-risk term / RSI penalty | **excluded** (objective harmonized to Z_ext) |

Frozen config: T0=800, alpha=0.98, gap_iter=20, 401 outer candidate proposals/search evaluations per call (NOT the total objective-function-call count -- see below).
Objective: Z_ext = D + 0.15T + 8L + 8O.


In [ ]:
import os, sys, json, itertools
import pandas as pd
assert 'REPO_ROOT' in dir(), "Run notebook 00 first."
sys.path.insert(0, REPO_ROOT)

with open(os.path.join(REPO_ROOT, "configs", "liu_alns_config.json")) as f:
    liu_config = json.load(f)
print(json.dumps(liu_config, indent=2))


## PUBLIC mode: statistical reproduction from de-identified corrected Liu-ALNS outputs

In [ ]:
DEID_DIR = os.path.join(REPO_ROOT, "data_deidentified")
EXP_DIR = os.path.join(DEID_DIR, "experiment_outputs")

if DATA_MODE == "public":
    liu_df = pd.read_csv(os.path.join(EXP_DIR, "liu_alns_block_level.csv"))
    print(f"Loaded de-identified corrected Liu-ALNS outcomes: {len(liu_df)} rows")

    n_blocks = liu_df.groupby(['delivery_date','trigger_fraction','shock','seed']).ngroups
    dup = liu_df.groupby(['delivery_date','trigger_fraction','shock','seed']).size()
    print(f"Unique blocks: {n_blocks} (expected 1,836)")
    print(f"Duplicates: {(dup>1).sum()} (expected 0)")
    if RUN_MODE == "full":
        assert n_blocks == 1836
        assert (dup>1).sum() == 0
    print("Completeness audit: PASS")


## PRIVATE + FULL mode: run the comparator (computational reproduction)

In [ ]:
if DATA_MODE == "private":
    from src.data import build_canonical_mapping, load_distance_matrix, load_travel_time_p50_matrix
    from src.simulator import SimulationContext
    from src.liu_alns import liu_alns_search
    from src.quantiles import P85_MULTIPLIER, P95_MULTIPLIER
    from src.checkpoint import load_done_keys, append_rows

    PRIVATE_DIR = os.path.join(REPO_ROOT, "data_private")
    if not os.path.exists(os.path.join(PRIVATE_DIR, "customer_day_stops_preprocessed.csv")):
        raise FileNotFoundError("DATA_MODE=private requires authorized operational inputs. See README Level 2.")

    mapping_df, customers, _ = build_canonical_mapping(os.path.join(PRIVATE_DIR, "customer_day_stops_preprocessed.csv"))
    parent_of = dict(zip(zip(mapping_df['delivery_date'], mapping_df['virtual_stop_id']), mapping_df['parent_physical_node']))
    dist_matrix = load_distance_matrix(os.path.join(PRIVATE_DIR, "combined_osrm_distance_matrix_km_long.csv"))
    p50_matrix = load_travel_time_p50_matrix(os.path.join(PRIVATE_DIR, "combined_travel_time_matrix_p50_long.csv"))
    ctx = SimulationContext(dist_matrix, p50_matrix, customers, parent_of, P85_MULTIPLIER, P95_MULTIPLIER,
                           sa_config['depot_start_min'], sa_config['operating_window_end_min'])

    print("Liu-ALNS engine ready (private/full computational mode).")
    print(f"max_iter={'5 (quick smoke test)' if RUN_MODE=='quick' else liu_config['max_iter']}, "
          f"T0={liu_config['T0']}, alpha={liu_config['alpha']}, gap_iter={liu_config['gap_iter']}")
    print("A full 1,836-block run follows the same checkpoint/resume pattern as notebook 06;")
    print("omitted here for brevity -- see notebook 06's PRIVATE-mode cell for the pattern")
    print("(same structure, calling src.liu_alns.liu_alns_search instead of src.search).")
else:
    print("DATA_MODE=public: skipping the computational re-run cell above.")


## Descriptive summary (regenerated from loaded outcomes)

In [ ]:
if DATA_MODE == "public":
    summary = dict(
        n=len(liu_df), lateness=liu_df['total_lateness_min'].mean(),
        otd_pct=100*liu_df['otd_num'].sum()/liu_df['otd_den'].sum(),
        rsi=liu_df['rsi'].mean(), moved=liu_df['moved_stops'].mean(),
        block_unchanged_pct=100*liu_df['block_fully_unchanged'].sum()/len(liu_df),
        no_harm_pct=100*(1-liu_df['block_any_harm'].sum()/len(liu_df)),
        runtime_ms_mean=liu_df['runtime_ms'].mean() if 'runtime_ms' in liu_df.columns else None,
    )
    print(json.dumps(summary, indent=2, default=str))
    results_dir = os.path.join(REPO_ROOT, "results")
    os.makedirs(results_dir, exist_ok=True)
    with open(os.path.join(results_dir, "liu_alns_descriptive_summary.json"), "w") as f:
        json.dump(summary, f, indent=2, default=str)


## Expected outputs / integrity checks

In [ ]:
checks = {}
if DATA_MODE == "public":
    checks["data_loaded"] = len(liu_df) > 0
    checks["no_duplicates"] = (dup>1).sum() == 0
    if RUN_MODE == "full":
        checks["exact_block_count"] = n_blocks == 1836
else:
    checks["private_mode_setup_ran"] = True

for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
NOTEBOOK_08_STATUS = "PASS" if all(checks.values()) else "FAIL"
print(f"\nNOTEBOOK 08 STATUS: {NOTEBOOK_08_STATUS}")
assert NOTEBOOK_08_STATUS == "PASS"
